# Session 4 · Lab C – DuckDB as a database file

**Runs locally** with your `.venv` kernel. Needs two months of taxi data and the zone lookup
(`python get_data.py --months 2`).

In Session 1 you used DuckDB **in memory**: `duckdb.connect()` and SQL straight on Parquet files.
Today we create a **persistent database** – a single file on disk, like SQLite, but built for analytics.

| | in-memory (`duckdb.connect()`) | database file (`duckdb.connect("taxi.duckdb")`) |
|---|---|---|
| Tables survive closing Python | ✗ | ✓ |
| Data stored in | RAM (spills to disk if needed) | one compressed, columnar file |
| Typical use | quick queries on files | a local analytics database / small warehouse |

In [ ]:
import os, sys, time, subprocess, glob
from pathlib import Path
import duckdb, pandas as pd

DATA = Path("../data")
DB = DATA / "taxi.duckdb"
FILES = sorted(glob.glob(str(DATA / "yellow_tripdata_2024-0[12].parquet")))   # two months
ZONES = DATA / "taxi_zone_lookup.csv"
print("DuckDB", duckdb.__version__, "|", len(FILES), "Parquet files | zones file:", ZONES.exists())

# start fresh every time you run the notebook top to bottom
for f in [DB, Path(str(DB) + ".wal")]:
    f.unlink(missing_ok=True)

## 1. Create the database
There is no server to install and no `CREATE DATABASE` statement: **connecting to a file that does not exist creates it.**

In [ ]:
con = duckdb.connect(str(DB))
print("file exists:", DB.exists(), f"| size {DB.stat().st_size/1e6:.2f} MB")

## 2. Load data into tables
`CREATE TABLE ... AS SELECT` (CTAS) reads the Parquet files once and stores the result **inside** the database file.
We keep the columns we need, rename the timestamps and compute the trip duration on the way in.

In [ ]:
t0 = time.perf_counter()
con.execute('''
    CREATE OR REPLACE TABLE trips AS
    SELECT
        VendorID,
        tpep_pickup_datetime  AS pickup_ts,
        tpep_dropoff_datetime AS dropoff_ts,
        passenger_count, trip_distance, PULocationID, DOLocationID,
        payment_type, fare_amount, tip_amount, total_amount,
        date_diff('second', tpep_pickup_datetime, tpep_dropoff_datetime) / 60.0 AS duration_min
    FROM read_parquet(?)
''', [FILES])
con.execute("CREATE OR REPLACE TABLE zones AS SELECT * FROM read_csv(?)", [str(ZONES)])
print(f"loaded in {time.perf_counter()-t0:.1f} s")

con.sql("SHOW TABLES").show()
con.sql("SELECT count(*) AS trips FROM trips").show()

In [ ]:
con.sql("DESCRIBE trips").show()

## 3. Veracity check in one line: `SUMMARIZE`
`SUMMARIZE` profiles every column: min, max, approx. unique values, null percentage, quartiles.

In [ ]:
con.sql("SUMMARIZE trips").df()[["column_name", "min", "max", "approx_unique", "null_percentage"]]

**Your turn (5 min):** `SUMMARIZE` shows suspicious minimum and maximum values. Write **one** query that counts,
in a single pass over `trips`:
- trips with `duration_min <= 0`,
- trips with `trip_distance = 0`,
- trips with `total_amount < 0`,
- trips with `passenger_count` missing.

Hint: `count(*) FILTER (WHERE ...)` or `sum(CASE WHEN ... THEN 1 ELSE 0 END)`.

In [ ]:
# TODO: one query, one pass over the table, four counts
con.sql('''
    SELECT
        count(*) AS all_trips
        -- , ...
    FROM trips
''').show()

## 4. Joins and views
A **view** stores the *query*, not the data. It is always up to date and costs no disk space.

In [ ]:
con.execute('''
    CREATE OR REPLACE VIEW trips_enriched AS
    SELECT t.*, pu.Borough AS pu_borough, pu.Zone AS pu_zone, dof.Borough AS do_borough
    FROM trips t
    JOIN zones pu  ON t.PULocationID = pu.LocationID
    JOIN zones dof ON t.DOLocationID = dof.LocationID
    WHERE t.duration_min BETWEEN 1 AND 180 AND t.trip_distance > 0
''')

con.sql('''
    SELECT pu_borough, count(*) AS trips,
           round(avg(duration_min), 1) AS avg_min,
           round(median(trip_distance), 2) AS median_miles
    FROM trips_enriched
    GROUP BY ALL
    ORDER BY trips DESC
''').show()

## 5. Close Python's connection – is the data still there?

In [ ]:
con.close()
print(f"database file: {DB.stat().st_size/1e6:.0f} MB  vs  Parquet input: {sum(os.path.getsize(f) for f in FILES)/1e6:.0f} MB")

con = duckdb.connect(str(DB))           # a brand-new connection
con.sql("SELECT table_name, table_type FROM information_schema.tables").show()

## 6. Transactions (ACID)
DuckDB is a real database: changes happen inside **transactions** and can be rolled back.

In [ ]:
n = lambda: con.execute("SELECT count(*) FROM trips").fetchall()[0][0]
print("before:          ", n())
con.execute("BEGIN TRANSACTION")
con.execute("DELETE FROM trips WHERE payment_type = 2")   # oops – all cash trips
print("inside the txn:  ", n())
con.execute("ROLLBACK")
print("after ROLLBACK:  ", n())

## 7. The limit: one writer at a time
DuckDB is an **embedded** database: it runs inside *your* Python process, there is no server in between.
The price: only **one process** may open the file for writing. We simulate a second user with a second Python process.

In [ ]:
other_user = f'''
import duckdb
try:
    duckdb.connect(r"{DB}", read_only=True).sql("SELECT count(*) FROM trips").show()
except Exception as e:
    print("second process failed:", type(e).__name__, "-", str(e).splitlines()[0][:120])
'''
print("1) we hold a READ-WRITE connection:")
print(subprocess.run([sys.executable, "-c", other_user], capture_output=True, text=True).stdout)

con.close()
con = duckdb.connect(str(DB), read_only=True)
print("2) we hold a READ-ONLY connection:")
print(subprocess.run([sys.executable, "-c", other_user], capture_output=True, text=True).stdout)

Many readers **or** one writer – fine for one analyst or one batch job, not for 50 dashboard users writing at once.
That is the moment for a database server (PostgreSQL) or, for really big data, a lakehouse with Spark.

## 8. Table vs. Parquet file – same query, two sources

In [ ]:
q_table = "SELECT hour(pickup_ts) AS h, avg(tip_amount) FROM trips WHERE payment_type = 1 GROUP BY h"
q_files = "SELECT hour(tpep_pickup_datetime) AS h, avg(tip_amount) FROM read_parquet(?) WHERE payment_type = 1 GROUP BY h"

def best(fn, r=3):
    ts = []
    for _ in range(r):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return min(ts)

print(f"table in .duckdb : {best(lambda: con.sql(q_table).fetchall()):.3f} s")
print(f"Parquet files    : {best(lambda: con.execute(q_files, [FILES]).fetchall()):.3f} s")

## 9. DuckDB talks to pandas, Polars and Arrow
Results convert with `.df()`, `.pl()` or `.arrow()`. DuckDB can also query a pandas DataFrame **by its variable name**.

In [ ]:
hourly = con.sql("SELECT hour(pickup_ts) AS hour, count(*) AS trips FROM trips GROUP BY ALL ORDER BY hour").df()
busy = hourly[hourly.trips > hourly.trips.median()]          # plain pandas

con.sql("SELECT min(hour) AS first_busy_hour, max(hour) AS last_busy_hour FROM busy").show()   # SQL on a pandas DF

## 10. Export results
**Your turn (5 min):** export the average duration and trip count per pickup borough and hour of day to
`../data/borough_hour.parquet` with `COPY (...) TO ... (FORMAT parquet)`. Read the file back with pandas to check it.

In [ ]:
con.close()
con = duckdb.connect(str(DB))      # read-write again, so COPY may write
out = DATA / "borough_hour.parquet"
# TODO: COPY ( SELECT ... FROM trips_enriched GROUP BY ... ) TO '...' (FORMAT parquet)

In [ ]:
con.close()
# The database lives in data/, which git ignores. Delete it when you no longer need it:
# DB.unlink()

## 11. Where does DuckDB fit?
| | DuckDB | SQLite | PostgreSQL | Spark |
|---|---|---|---|---|
| Runs as | library in your process | library in your process | server | cluster (or local) |
| Storage layout | **columnar** | row | row | files (Parquet/Delta) |
| Built for | analytics (OLAP) | transactions in apps (OLTP) | OLTP + some OLAP | analytics on many machines |
| Concurrent writers | 1 process | 1 at a time | many | many jobs on files |
| Setup | `pip install duckdb` | built into Python | install + admin | JVM + cluster |

## 12. Exam-style questions
1. Explain the difference between `duckdb.connect()` and `duckdb.connect("taxi.duckdb")`. Give one situation for each.
2. Why is DuckDB called an *embedded* database? Name one advantage and one limitation that follow from this.
3. Scenario: a team of 30 analysts wants a shared dashboard that writes user annotations back to the database
   every few seconds. Is a single DuckDB file a good choice? Justify and propose an alternative.
4. What is the difference between a `TABLE` created with `CREATE TABLE ... AS SELECT` and a `VIEW`? What happens to each
   when new rows are inserted into `trips`?
5. You run `DELETE FROM trips WHERE payment_type = 2` inside `BEGIN TRANSACTION` and then `ROLLBACK`. How many rows
   does `trips` have afterwards, and which ACID property does this demonstrate?
6. Compare the size of your `.duckdb` file with the Parquet input. Why are both in the same order of magnitude,
   while a CSV export of the same table would be several times larger?